# Online Shoppers Purchasing Intention: Week 2 (Load + EDA)

**Business question:** which session behaviours predict a purchase (`Revenue`)?

**Data:** Sakar, C. and Kastro, Y. (2018) *Online Shoppers Purchasing Intention Dataset*. UCI ML Repository. https://doi.org/10.24432/C5F88Q (CC BY 4.0)

## 1. Imports and settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Load data

In [ ]:
DATA_PATH = '../data/raw/online_shoppers_intention.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Structure: shape, types, missing values, duplicates\n*Why:* confirms what we are working with and what cleaning is needed.

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 4. Summary statistics\n*Why:* mean, std and quartiles show scale differences (needed for scaling) and skew/outliers.

In [ ]:
df.describe().T

In [ ]:
# Variance explicitly (brief asks for central tendency + spread)
num_cols = df.select_dtypes(include='number').columns
pd.DataFrame({'mean': df[num_cols].mean(), 'median': df[num_cols].median(),
              'variance': df[num_cols].var(), 'std': df[num_cols].std()})

## 5. Target distribution\n*Why:* class imbalance decides which metrics and sampling strategy we justify later.

In [ ]:
print(df['Revenue'].value_counts())
print(df['Revenue'].value_counts(normalize=True).round(3))
sns.countplot(x='Revenue', data=df)
plt.title('Revenue (purchase) distribution')
plt.show()

## 6. Categorical features vs target\n*Why:* shows which groups convert more.

In [ ]:
cat_cols = ['Month', 'VisitorType', 'Weekend']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    rate = df.groupby(col)['Revenue'].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.set_title(f'Purchase rate by {col}')
    ax.set_ylabel('Share with Revenue=True')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print(sorted(df['Month'].unique()))  # note: check naming (e.g. 'June' vs 'Jun') and missing months

## 7. Distributions of numeric features\n*Why:* the duration/count columns are usually heavily right-skewed; this motivates log transform or scaling.

In [ ]:
num_plot = ['Administrative_Duration', 'Informational_Duration', 'ProductRelated',
            'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues']
df[num_plot].hist(bins=40, figsize=(15, 8))
plt.tight_layout(); plt.show()

## 8. Outliers (box plots by target)\n*Why:* decide whether to cap, transform or keep outliers.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_plot):
    sns.boxplot(x='Revenue', y=col, data=df, ax=ax)
    ax.set_title(col)
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

## 9. Correlation matrix\n*Why:* finds redundant features (e.g. BounceRates vs ExitRates) and the strongest links to Revenue.

In [ ]:
corr = df.select_dtypes(include=['number', 'bool']).astype(float).corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()
corr['Revenue'].drop('Revenue').sort_values(ascending=False)

## 10. Scatter plot: engagement vs PageValues\n*Why:* `PageValues` is the strongest predictor; check how it relates to other behaviour.

In [ ]:
sns.scatterplot(data=df, x='ProductRelated_Duration', y='PageValues', hue='Revenue', alpha=0.4)
plt.title('ProductRelated_Duration vs PageValues')
plt.show()

## 11. EDA findings (fill in after running)\n- Class imbalance:\n- Duplicates:\n- Skew / outliers:\n- Strongest correlations:\n- Month / VisitorType effects:\n- Data quality issues to fix in Week 3: